In [106]:
#import packages
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, make_scorer
from sklearn.model_selection import cross_val_score
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings
import sys
sys.path.append(r".venv\Lib\site-packages")
import pycountry

#cases_df = pd.read_csv("weekly AFR cases by country as of 19 January 2025(in).csv")
#print(cases_df.head())

In [107]:
# clean logistic model outbreak data

logistic_outbreaks_df = pd.read_csv("fit_data/76cases-based-on-consecutive3-week-no-increase/logistic_fitted_parameters_all_countries.csv")
#print(logistic_outbreaks_df.head())
logistic_outbreaks_df['start_date'] = pd.to_datetime(logistic_outbreaks_df['start_date'])
logistic_outbreaks_df['start_year'] = logistic_outbreaks_df['start_date'].dt.year

lo = logistic_outbreaks_df[['country_code', 'start_year', 'termination_reason', 'goodness_of_fit', 'K', 'r', 't0']].copy()
# Map termination_reason to numeric values
termination_mapping = {"4 consecutive weeks no increase": 0, "reached last data": 1}  # Adjust if needed
lo["termination_reason"] = lo["termination_reason"].map(termination_mapping)

print(lo.head(10))

# add external factors


  country_code  start_year  termination_reason  goodness_of_fit            K  \
0          AGO        2024                   0         0.997743     4.037147   
1          BEN        2022                   0         0.999985     3.003464   
2          BDI        2024                   1         0.999125  3321.021891   
3          CIV        2024                   0         0.997290   107.932901   
4          CMR        2022                   0         0.997728     3.024118   
5          CMR        2022                   0         0.999985     1.001155   
6          CMR        2022                   0         0.984421     4.136732   
7          CMR        2022                   0         0.989819    10.454969   
8          CMR        2023                   0         0.984774    24.382151   
9          CMR        2023                   0         0.999985     1.001155   

           r         t0  
0   1.347953   5.020870  
1  10.000000   3.504546  
2   0.247075  19.828712  
3   0.286981  1

In [ ]:
#cleaning cases data
# Select relevant columns
cases_df_model = cases_df[['country', 'week_end_date', 'total_confirmed_cases', 'new_confirmed_cases']].copy()
cases_df_model['week_end_date'] = pd.to_datetime(cases_df_model['week_end_date'])

cases_df_model = cases_df_model.sort_values(by=['country', 'week_end_date'], ascending=[True, True])

# Create lag feature for new weekly cases
cases_df_model['previous_week_cases'] = cases_df_model.groupby('country', group_keys=False)['total_confirmed_cases'].shift(1)

# Update column safely without using inplace=True
cases_df_model['previous_week_cases'] = cases_df_model['previous_week_cases'].fillna(0)

cases_df_model['month'] = cases_df_model['week_end_date'].dt.month  # seasonality effect

#cases_df_model.iloc[170:191]

In [ ]:
# healthcare_df = pd.read_csv("Healthcare expenditure data as percentage of GDP (%).csv")
# healthcare_df.head()

In [108]:
#population of each country
warnings.filterwarnings("ignore")  # suppress warnings

population_df = pd.read_csv("f4b22788-e8b8-400a-b828-fb47aa226b25_Series - Metadata.csv")
population_df.head()

# filter population data to include only the countries in cases data
countries_in_lo = lo['country_code'].unique()
population_df_filtered = population_df[population_df['Country Code'].isin(countries_in_lo)]
population_df_filtered = population_df_filtered.iloc[:, 3:]
population_df_filtered.columns = population_df_filtered.columns.str.split(' ').str[0]
population_df_filtered.rename(columns={"Country": "country_code"}, inplace=True)


#Evaluate ARIMA model
# Define sMAPE function
def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=np.float64), np.array(y_pred, dtype=np.float64)  # Ensure numeric
    return np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100

# Lists to store actual and predicted values
actual_values = []
predicted_values = []

train_year_columns = population_df_filtered.columns[1:-2].astype(int)

# Loop over each country
for index, row in population_df_filtered.iterrows():
    country_code = row['country_code']
    
    # Use data from 1990 to 2021 for training
    train_data = row[1:-2].values.astype(float)  # Data for years 1990-2021
    train_data_series = pd.Series(train_data, index=train_year_columns)
    
    # Actual values for 2022 and 2023
    actual_2022 = row['2022']
    actual_2023 = row['2023']
    
    try:
        # Fit ARIMA model
        model = ARIMA(train_data_series, order=(1, 1, 1))  # ARIMA(1,1,1) example
        model_fit = model.fit()
        
        # Forecast for 2022 and 2023
        forecast = model_fit.forecast(steps=2)
        
        # Collect predictions and actual values
        predicted_2022, predicted_2023 = forecast
        actual_values.extend([actual_2022, actual_2023])
        predicted_values.extend([predicted_2022, predicted_2023])
        
    except Exception as e:
        print(f"Failed for {country_code}: {e}")

# Compute overall error metrics
overall_mae = mean_absolute_error(actual_values, predicted_values)
overall_mse = mean_squared_error(actual_values, predicted_values)
overall_rmse = np.sqrt(overall_mse)

print(f"Overall MAE: {overall_mae}")
print(f"Overall MSE: {overall_mse}")
print(f"Overall RMSE: {overall_rmse}")
#print(actual_values)
#print(predicted_values)

# Compute overall MAPE and sMAPE
overall_mape = mean_absolute_percentage_error(actual_values, predicted_values) * 100
overall_smape = smape(np.array(actual_values), np.array(predicted_values))

print(f"Overall MAPE: {overall_mape:.2f}%")
print(f"Overall sMAPE: {overall_smape:.2f}%")



Overall MAE: 355854.9170963876
Overall MSE: 343950885096.6425
Overall RMSE: 586473.2603423983
Overall MAPE: 1.18%
Overall sMAPE: 1.20%


In [109]:
#predict 2024, 2025

year_columns = population_df_filtered.columns[1:].astype(int)
latest_year = np.int64(2023)

# add empty columns for 2024 and 2025
population_df_filtered['2024'] = None
population_df_filtered['2025'] = None

# loop over each country
for index, row in population_df_filtered.iterrows():
    populations = row[1:len(year_columns)+1].values.astype(float)
    
    population_series = pd.Series(populations, index=year_columns)
    
    try:
        # fit fixed ARIMA(1,1,1) model
        model = ARIMA(population_series, order=(1, 1, 1))
        model_fit = model.fit()
        
        # forecast next 2 years
        forecast = model_fit.forecast(steps=2)
        
        # store forecasts
        population_df_filtered.at[index, '2024'] = round(forecast.iloc[0])
        population_df_filtered.at[index, '2025'] = round(forecast.iloc[1])
        
    except Exception as e:
        print(f"Failed for {row['Country']}: {e}")
        population_df_filtered.at[index, '2024'] = None
        population_df_filtered.at[index, '2025'] = None

population_df_filtered.tail()
pop_df = population_df_filtered
pop_df = pop_df.reset_index(drop=True)
print(len(pop_df))
pop_df.head()

25


,country_code,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,AGO,11626360,12023529,12423712,12827135,13249764,13699778,14170973,14660413,15159370,...,29183070,30234839,31297155,32375632,33451132,34532429,35635029,36749906,37510529,38271142
1,BEN,5281479,5446932,5617844,5872706,6096506,6226773,6391858,6583365,6789489,...,11697842,12039780,12383347,12726755,13070169,13413417,13759501,14111034,14375550,14640062
2,BDI,5587052,5703857,5858407,5676843,5713854,6066316,6070506,6070042,6187108,...,11239451,11506762,11859446,12255336,12617036,12965481,13321097,13689450,13933055,14176658
3,CMR,11331821,11667180,12006353,12353131,12704903,13058516,13414757,13775214,14144860,...,23454161,24128601,24806383,25506095,26210558,26915758,27632771,28372687,28889506,29406307
4,CAF,2871910,2963191,3058831,3157839,3257954,3348052,3435965,3531700,3628827,...,4713663,4793511,4878657,4944703,5026628,5112100,5098039,5152421,5192241,5230872


In [ ]:
# year_columns = [col for col in pop_df.columns if col.isdigit()]
# print(year_columns)
# for i in pop_df['country_code']:
#     print(i)
#     print(type(i))
# print('AGO' in pop_df['country_code'])
# print(any(pop_df['country_code'] == 'AGO'))
# print(pop_df.loc[pop_df['country_code']=='AGO'])
# print(pop_df.loc[pop_df['country_code']=='AGO', '2020'])
# print(pop_df.loc[pop_df['country_code']=='BEN', '2020'])
# a=pop_df.loc[pop_df['country_code']=='BEN', '2020'].iloc[0]
# print(a)
#pop_df.loc[pop_df['country_code']=='AFG', '1990'].iloc[0]

In [110]:
#GDP per capita
GDPpc_df = pd.read_csv("GDPpercapitadata .csv")

def get_country_code(country_name):
    try:
        if country_name == "Congo, Dem. Rep. of the":
            return 'COD'
        elif country_name == "Congo, Republic of ":
            return 'COG'
        return pycountry.countries.lookup(country_name).alpha_3  # Get 3-letter code (ISO Alpha-3)
    except LookupError:
        return None  # Return None if country not found

country_list = GDPpc_df['Countries']
country_codes = [get_country_code(country) for country in country_list]

GDPpc_df.insert(GDPpc_df.columns.get_loc('Countries') + 1, 'country_code', country_codes)

GDPpc_df.head()

# filter GDP data to include only the countries in cases data
gdp_df_filtered = GDPpc_df[GDPpc_df['country_code'].isin(countries_in_lo)]
gdp_df_filtered = gdp_df_filtered.iloc[:, 1:]
print(len(gdp_df_filtered))
gdp_df_filtered.head()

# Find elements in df1 but not in df2
diff_values = set(pop_df['country_code']) - set(gdp_df_filtered['country_code'])
print(diff_values)

gdp_df = gdp_df_filtered
gdp_df = gdp_df.reset_index(drop=True)
gdp_df.head(10)


25
set()


,country_code,1980,1981,1982,1983,1984,1985,1986,1987,1988,...,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
0,AGO,1055.339,961.721,936.935,951.5,982.624,1074.327,977.905,1089.96,1151.19,...,1989.939,2445.388,3974.666,2967.384,2961.34,2990.597,3030.019,3140.94,3222.419,3316.369
1,BEN,614.04,401.43,385.574,330.58,341.857,359.165,448.478,519.04,527.911,...,1239.74,1361.816,1306.021,1433.067,1510.188,1586.803,1667.858,1752.074,1836.895,1922.215
2,BDI,232.381,234.94,241.083,247.995,218.999,247.644,253.424,231.932,211.184,...,259.906,273.956,310.952,326.842,320.636,156.497,168.286,179.305,190.883,200.024
3,CMR,1004.846,1106.193,1031.862,1012.33,1041.933,1059.933,1344.534,1514.218,1493.9,...,1539.354,1654.245,1592.262,1722.821,1821.317,1922.852,2018.241,2106.939,2208.328,2315.635
4,CAF,321.769,324.98,306.755,287.218,272.557,349.348,442.905,477.652,505.629,...,475.188,505.678,482.523,510.568,529.492,548.832,573.245,593.472,613.025,633.676
5,COD,2730.694,2301.277,2422.242,1891.283,1221.72,1080.898,1174.016,1075.998,1206.847,...,546.43,614.306,679.412,669.581,702.373,743.648,767.704,793.139,811.051,829.704
6,COG,1268.332,1008.636,853.044,752.384,671.076,667.569,885.759,1043.902,1077.321,...,2014.193,2293.987,2339.437,2308.52,2384.399,2454.478,2522.121,2587.661,2670.943,2781.53
7,CIV,1764.741,1429.982,1235.211,1060.684,1028.742,1010.216,1301.055,1377.855,1379.623,...,2209.109,2478.224,2326.334,2537.065,2719.971,2901.813,3071.685,3268.378,3449.146,3632.39
8,EGY,580.042,617.659,711.351,846.388,925.97,1049.33,1132.456,1585.217,1858.029,...,3802.438,4145.939,4587.172,3743.608,3541.75,3160.11,3468.707,3786.363,4132.243,4502.88
9,GAB,6093.615,5376.616,4930.527,4619.015,4346.388,4478.468,5735.355,4245.955,4581.144,...,7289.442,9113.798,9478.236,9079.278,9256.657,9094.46,9127.831,9168.27,9230.524,9308.535


In [111]:
#sanitation
sanitation_df = pd.read_csv("Mpox____sanitation____.csv")

def get_country_code(country_name):
    try:
        if country_name == "Democratic Republic of the Congo":
            return 'COD'
        elif country_name == "Congo, Republic of ":
            return 'COG'
        elif country_name == "Côte d’Ivoire":
            return 'CIV'
        return pycountry.countries.lookup(country_name).alpha_3  # Get 3-letter code (ISO Alpha-3)
    except LookupError:
        return None  # Return None if country not found

san_country_list = sanitation_df['country']
san_country_codes = [get_country_code(country) for country in san_country_list]

sanitation_df.insert(sanitation_df.columns.get_loc('year') + 1, 'country_code', san_country_codes)

sanitation_df.head()

# filter sanitation data to include only the countries in cases data
sanitation_df_filtered = sanitation_df[sanitation_df['country_code'].isin(countries_in_lo)]
sanitation_df_filtered = sanitation_df_filtered.iloc[:, 3:]
print(len(sanitation_df_filtered))
sanitation_df_filtered.head()

# Find elements in df1 but not in df2
diff_values = set(pop_df['country_code']) - set(sanitation_df_filtered['country_code'])
print(diff_values)

print(sanitation_df_filtered.head())

san_df = sanitation_df_filtered
san_df = san_df.reset_index(drop=True)
san_df.head()


25
set()
  country_code  basic_sanitation  limited_sanitation  unimproved_sanitation  \
0          AGO         52.177276           21.279858               9.254820   
1          BEN         19.493502           19.966929              12.037360   
2          BDI         45.690685           12.906042              39.958334   
3          CMR         43.119165           17.051006              35.579366   
4          CAF         13.846294           16.313945              44.795126   

   open_defecation  
0        17.288046  
1        48.502210  
2         1.444939  
3         4.250463  
4        25.044635  


,country_code,basic_sanitation,limited_sanitation,unimproved_sanitation,open_defecation
0,AGO,52.177276,21.279858,9.254820,17.288046
1,BEN,19.493502,19.966929,12.037360,48.502210
2,BDI,45.690685,12.906042,39.958334,1.444939
3,CMR,43.119165,17.051006,35.579366,4.250463
4,CAF,13.846294,16.313945,44.795126,25.044635


In [112]:
lo["start_year"] = lo["start_year"].astype(str)
year_columns = [col for col in pop_df.columns if col.isdigit()]
lo['population'] = lo['country_code'].apply(lambda code: pop_df.loc[pop_df['country_code']==code, lo.loc[lo['country_code']== code , 'start_year'].iloc[0]].iloc[0])
lo['gdp_per_capita'] = lo['country_code'].apply(lambda code: gdp_df.loc[gdp_df['country_code']==code, lo.loc[lo['country_code']== code , 'start_year'].iloc[0]].iloc[0])
lo = lo.merge(san_df, on='country_code', how='left')

In [ ]:
print(lo.head())
#termination reason, 0:ended after 3 weeks same, 1:reached end of data

  country_code start_year  termination_reason  goodness_of_fit            K  \
0          AGO       2024                   0         0.997743     4.037147   
1          BEN       2022                   0         0.999985     3.003464   
2          BDI       2024                   1         0.999125  3321.021891   
3          CIV       2024                   0         0.997290   107.932901   
4          CMR       2022                   0         0.997728     3.024118   

           r         t0 population gdp_per_capita  basic_sanitation  \
0   1.347953   5.020870   37510529        2961.34         52.177276   
1  10.000000   3.504546   13759501       1306.021         19.493502   
2   0.247075  19.828712   13933055        320.636         45.690685   
3   0.286981  14.033539   31739421       2719.971         37.000000   
4   1.828490   4.513635   27632771       1592.262         43.119165   

   limited_sanitation  unimproved_sanitation  open_defecation  
0           21.279858             

In [118]:
# Step 1: Have a data frame with relevant factors
df = lo.copy()

# Step 2: Define features (x) and target (y)
x = df[['population', 'gdp_per_capita', 'basic_sanitation', 'limited_sanitation', 'unimproved_sanitation', 'open_defecation', 'termination_reason']]
y = df[['K', 'r', 't0']]

# Step 3: Split data into training (80%) and testing (20%)
n = 10
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)

# Step 4: Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=n)
rf.fit(x_train, y_train)

# Step 5: Compute Feature Importance
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Step 6: Make Predictions on Test Data
y_pred = rf.predict(x_test)

# Step 7: Evaluate Model
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_test, y_pred)

def smape(y_true, y_pred):
    return np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100

overall_smape = smape(y_test, y_pred)

print(f'MAE: {mae:.2f}')
print(f'MSE: {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'sMAPE: {overall_smape:.2f}%')

# Step 8: Predict Future (e.g. 2025) Monkeypox Cases
# function to predict
# def predict_cases_for_date(last_known_date, total_cases_so_far, last_week_cases, future_date):
#     last_known_date = pd.to_datetime(last_known_date)
#     future_date = pd.to_datetime(future_date)
    
#     if future_date <= last_known_date:
#         print("Error: Future date must be after the last known date.")
#         return None
    
#     num_weeks = (future_date - last_known_date).days // 7
    
#     predicted_total_cases = total_cases_so_far
#     previous_week_cases = last_week_cases
#     current_date = last_known_date

#     for i in range(num_weeks):
#         current_date_num = current_date.toordinal()
#         month = current_date.month
#         new_confirmed_cases = predicted_total_cases - previous_week_cases
        
#         new_data = pd.DataFrame({
#             'week_end_date_num': [current_date_num],
#             'total_confirmed_cases': [predicted_total_cases],
#             'previous_week_cases': [previous_week_cases],
#             'new_confirmed_cases': [new_confirmed_cases],
#             'month': [month],
#         })
        
#         # Predict this week's total cases
#         predicted_new_cases = rf.predict(new_data)[0]
        
#         # Update for the next iteration
#         previous_week_cases = predicted_total_cases
#         predicted_total_cases += predicted_new_cases
#         current_date += pd.Timedelta(weeks=1)
    
#     print(f"Predicted Monkeypox Cases for {future_date.date()}: {predicted_total_cases:.0f}")

def predict_parameters(population, gdp_per_capita, basic_sanitation, limited_sanitation, unimproved_sanitation, open_defecation, termination_reason):
    new_data = pd.DataFrame({
        'population': [population],
        'gdp_per_capita': [gdp_per_capita],
        'basic_sanitation' : [basic_sanitation],
        'limited_sanitation' : [limited_sanitation],
        'unimproved_sanitation' : [unimproved_sanitation],
        'open_defecation' : [open_defecation],
        'termination_reason': [termination_reason]})
    
    # Predict parameters
    predicted_para = rf.predict(new_data)
    
    print(f"Predicted parameters for:")
    print(f"population={population}")
    print(f"GDP per capita={gdp_per_capita}")
    print(f"basic sanitation={basic_sanitation}")
    print(f"limited sanitation={limited_sanitation}")
    print(f"unimproved sanitation={unimproved_sanitation}") 
    print(f"open defecation={open_defecation}") 
    print(f"termination reason={termination_reason}")
    print(f"prediction: K:{predicted_para[0][0]}, r:{predicted_para[0][1]}, t0:{predicted_para[0][2]}")

Feature Importance:
                 Feature  Importance
4  unimproved_sanitation    0.475198
6     termination_reason    0.240671
0             population    0.166927
5        open_defecation    0.039449
1         gdp_per_capita    0.033918
2       basic_sanitation    0.027242
3     limited_sanitation    0.016594
MAE: 93.54
MSE: 231422.99
RMSE: 481.06
MAPE: 9.10%
sMAPE: 100.00%


In [119]:
#predict_cases_for_date('2025-01-05', 3035, 2946, '2025-01-12')
predict_parameters(37510529, 2961.34, 52.177276, 21.279858, 9.254820, 17.288046, 0)

Predicted parameters for:
population=37510529
GDP per capita=2961.34
basic sanitation=52.177276
limited sanitation=21.279858
unimproved sanitation=9.25482
open defecation=17.288046
termination reason=0
prediction: K:5.746620048018878, r:3.7569462901897204, t0:4.862906937695309


In [125]:
# Perform K-fold cross-validation (with 5 folds in this example)
cv_scores = cross_val_score(rf, x, y, cv=5, scoring='neg_mean_squared_error')

# Convert negative MSE to positive and print the results
cv_scores = -cv_scores  # MSE is negative in scikit-learn for regression
print(f"Cross-Validation MSE scores: {cv_scores}")
print(f"Mean CV MSE: {cv_scores.mean()}")

#instead of using mean squared error, use percentage error
# Define MAPE scorer (scikit-learn negates by default, so we use a custom scorer)
mape_scorer = make_scorer(mean_absolute_percentage_error, greater_is_better=False)

# Perform K-fold cross-validation using MAPE
cv_mape_scores = cross_val_score(rf, x, y, cv=5, scoring=mape_scorer)

# Convert negative MAPE to positive
cv_mape_scores = -cv_mape_scores * 100  # Convert to percentage

# Print results
print(f"Cross-Validation MAPE scores: {cv_mape_scores}")
print(f"Mean CV MAPE: {cv_mape_scores.mean():.2f}%")

#use smape
def smape(y_true, y_pred):
    return np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100

smape_scorer = make_scorer(smape, greater_is_better=False)

cv_smape_scores = cross_val_score(rf, x, y, cv=5, scoring=smape_scorer)

cv_smape_scores = -cv_smape_scores  # Convert to positive values

print(f"Cross-Validation sMAPE scores: {cv_smape_scores}")
print(f"Mean CV sMAPE: {cv_smape_scores.mean():.2f}%") 

Cross-Validation MSE scores: [2.23551017e+05 2.76538662e+06 2.00905017e+07 9.83078516e+03
 4.97350133e+06]
Mean CV MSE: 5612554.297926882
Cross-Validation MAPE scores: [ 228.40070821 4085.6131439   803.40974998 2155.04837008  601.56020796]
Mean CV MAPE: 1574.81%
Cross-Validation sMAPE scores: [ 69.47058414  94.44310499  97.4333603  100.85951511  98.11789293]
Mean CV sMAPE: 92.06%


In [ ]:
# Step 1: Have a data frame with data of the different factors we want to look at
# e.g. df = pd.DataFrame()

# (Step 1.5: Could create lag features for previous year, for the model to use previous year data as another factor)
# e.g. df['Lag_Healthcare_Expenditure'] = df['Healthcare_Expenditure'].shift(1)

# Step 2: Define factors (x) and target (y)
# e.g. x = df[['Year', 'Lag_Healthcare_Expenditure', 'Population_Density']]
#      y = df['Monkeypox_Cases']

# Step 4: Split data into training and testing (e.g. 80% train, 20% test)
# n = 10
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)
# random state to make sure results are repeatable

# Step 5: Train the Random Forest Model
# e.g. rf = RandomForestRegressor(n_estimators=100, random_state=n)
# rf.fit(x_train, y_train)

# Step 6: Compute Feature Importance
# feature_importance = pd.DataFrame({
#     'Feature': x.columns,
#     'Importance': rf.feature_importances_
# }).sort_values(by='Importance', ascending=False)

# Display Feature Importance
# print("Feature Importance:")
# print(feature_importance)

# Step 7: Make Predictions
# y_pred = rf.predict(x_test)

# Step 8: Evaluate the Model
# mae = mean_absolute_error(y_test, y_pred)
# print(f'Mean Absolute Error: {mae:.2f}')

# Step 9: Predict Future (e.g. 2025) Monkeypox Cases
# e.g. new_data = pd.DataFrame({
#     'Year': [2025],
#     'Lag_Healthcare_Expenditure': [5000],  # Previous year's value (2024)
#     'Lag_Population_Density': [150],  # Previous year's value (2024)
# })

# prediction_2025 = rf.predict(new_data)
# print(f'Predicted Monkeypox Cases for 2025: {prediction_2025[0]:.0f}')